In [ ]:
import gdown, os
os.makedirs("data", exist_ok=True)

files = {
    "data/sample_submission.csv": "1FzylddvJTSACVs7zWP8qWHUVi-r_rqT5",
    "data/test_data.parquet":     "1JV2c288TkCSsa7k1HaTGgteC_eT12rYi",
    "data/train_data.parquet":    "1qlJ9-HuTKaqRv8AqOYBCsPcTk2A-YWXT",
    "data/train_target.csv":      "1Q_phWCv7-_PwlDadbEBRLWYZmrD9bLVK",
}

for path, fid in files.items():
    print(f"Downloading {path}...")
    gdown.download(id=fid, output=path, quiet=False)

print("\nДанные:")
for path in files:
    size = os.path.getsize(path) / 1e6
    print(f"  {path:35s} {size:.1f} MB")

In [ ]:
!pip install lightgbm tqdm -q

import os, gc, glob, time, warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
import lightgbm as lgb
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/working/data"
WORK_DIR = "/kaggle/working"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")
print(f"Data   : {DATA_DIR}")

In [ ]:
def load_parts(stem):
    files = sorted(glob.glob(f"{DATA_DIR}/{stem}*.parquet"))
    if not files:
        files = sorted(glob.glob(f"{DATA_DIR}/**/{stem}*.parquet", recursive=True))
    frames = []
    for f in tqdm(files, desc=stem, leave=False):
        ch = pd.read_parquet(f)
        for c in ch.columns:
            if c == "id":
                continue
            if pd.api.types.is_integer_dtype(ch[c]):
                ch[c] = pd.to_numeric(ch[c], downcast="integer")
            elif pd.api.types.is_float_dtype(ch[c]):
                ch[c] = pd.to_numeric(ch[c], downcast="float")
        frames.append(ch)
    df = pd.concat(frames, ignore_index=True)
    del frames; gc.collect()
    mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"{stem:12s}  rows={len(df):>10,}  cols={df.shape[1]}  {mb:.0f} MB")
    return df

train  = load_parts("train_data")
test   = load_parts("test_data")
target = pd.read_csv(f"{DATA_DIR}/train_target.csv")
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print(f"\nDefault rate: {target['flag'].mean():.2%}")

In [ ]:
def paym_features(df):
    paym_cols = sorted(
        [c for c in df.columns if c.startswith("enc_paym_")],
        key=lambda x: int(x.split("_")[-1])
    )
    if not paym_cols:
        return pd.DataFrame(index=df["id"].unique())
    n = len(paym_cols)
    g = df.groupby("id", sort=True)
    parts = []

    mx = g[paym_cols].max().astype("int8")
    parts.append(mx.max(axis=1).rename("paym_max_status").astype("int8"))

    bad_sum = pd.Series(0, index=mx.index, dtype="int16")
    for c in tqdm(paym_cols, desc="paym bad", leave=False):
        tmp = pd.DataFrame({"id": df["id"].values, "b": (df[c] >= 2).astype("int8").values})
        bad_sum = bad_sum.add(tmp.groupby("id")["b"].sum(), fill_value=0)
        del tmp

    n_prod = g.size().astype("int16")
    parts.append((bad_sum / (n * n_prod)).astype("float16").rename("paym_bad_rate"))
    parts.append(bad_sum.astype("int16").rename("paym_bad_cnt"))
    del bad_sum; gc.collect()

    for k, alias in [(1,"m0"),(3,"last3"),(6,"last6"),(12,"last12")]:
        w = paym_cols[:min(k, n)]
        bw = pd.Series(0, index=mx.index, dtype="int16")
        for c in w:
            tmp = pd.DataFrame({"id": df["id"].values, "b": (df[c] >= 2).astype("int8").values})
            bw = bw.add(tmp.groupby("id")["b"].sum(), fill_value=0)
            del tmp
        parts.append(bw.astype("int8").rename(f"paym_{alias}"))
        del bw; gc.collect()

    c0 = paym_cols[0]
    t0 = pd.DataFrame({"id": df["id"].values, "s": df[c0].values})
    g0 = t0.groupby("id")["s"]
    parts += [g0.mean().astype("float16").rename("paym_m0_mean"),
              g0.max().astype("int8").rename("paym_m0_max")]
    del t0, g0, mx; gc.collect()

    out = pd.concat(parts, axis=1)
    out.index.name = "id"
    return out


def build_features(df):
    feat_cols = [c for c in df.columns if c not in ("id","rn")]
    enc_cols  = [c for c in feat_cols if c.startswith("enc_")]
    paym_cols = [c for c in enc_cols  if c.startswith("enc_paym_")]
    cat_cols  = [c for c in enc_cols  if not c.startswith("enc_paym_")]
    ord_cols  = [c for c in feat_cols if not c.startswith("enc_")]
    g         = df.groupby("id", sort=True)
    parts     = []

    with tqdm(total=7, desc="features", leave=False) as bar:

        parts.append(pd.DataFrame({
            "hist_n":       g.size().astype("int16"),
            "hist_rn_last": g["rn"].max().astype("int16"),
        }))
        bar.update()

        agg = g[feat_cols].agg(["mean","max","min"])
        agg.columns = [f"{c}_{f}" for c,f in agg.columns]
        parts.append(agg.astype("float16"))
        del agg; gc.collect()
        bar.update()

        rn_max   = df.groupby("id")["rn"].transform("max")
        df_l3    = df.loc[df["rn"] >= rn_max - 2, ["id"] + ord_cols]
        agg_l3   = df_l3.groupby("id", sort=True)[ord_cols].agg(["mean","max"])
        agg_l3.columns = [f"{c}_l3_{f}" for c,f in agg_l3.columns]
        parts.append(agg_l3.astype("float16"))
        del df_l3, rn_max, agg_l3; gc.collect()
        bar.update()

        li = g["rn"].idxmax(); fi = g["rn"].idxmin()
        last  = df.loc[li, ["id"]+feat_cols].set_index("id").add_suffix("_last")
        first = df.loc[fi, ["id"]+feat_cols].set_index("id").add_suffix("_first")
        for fr in (last, first):
            for c in fr.columns:
                if pd.api.types.is_integer_dtype(fr[c]):
                    fr[c] = pd.to_numeric(fr[c], downcast="integer")
        parts += [last, first]
        del last, first, li, fi; gc.collect()
        bar.update()

        wc = [c for c in ord_cols if any(k in c for k in
              ("loans9","loans6","loans3","loans5","overdue","over2limit","util"))]
        if wc:
            parts.append(g[wc].max().add_suffix("_worst").astype("float16"))
        for col, alias in [("is_zero_loans90","share90"),
                            ("is_zero_loans_6090","share6090"),
                            ("is_zero_loans_3060","share3060")]:
            if col in df.columns:
                tmp = pd.DataFrame({"id":df["id"].values,
                                    "v":(df[col]==0).astype("int8").values})
                parts.append(tmp.groupby("id")["v"].mean()
                             .astype("float16").rename(alias))
                del tmp
        gc.collect()
        bar.update()

        parts.append(paym_features(df))
        gc.collect()
        bar.update()

        for c in tqdm(cat_cols, desc="bag-of-cats", leave=False):
            if df[c].nunique(dropna=True) <= 30:
                ct = pd.crosstab(df["id"], df[c]).clip(upper=255).astype("uint8")
                ct.columns = [f"{c}_v{int(v)}_cnt" for v in ct.columns]
                parts.append(ct); del ct; gc.collect()
        bar.update()

    out = pd.concat(parts, axis=1)
    del parts; gc.collect()
    out.index.name = "id"
    mb = out.memory_usage(deep=True).sum() / 1e6
    print(f"  features: {out.shape[1]} cols  {mb:.0f} MB")
    return out


print("Building train features...")
Xtr = build_features(train); del train; gc.collect()
print("Building test features...")
Xte = build_features(test);  del test;  gc.collect()

Xte = Xte.reindex(columns=Xtr.columns)
cnt = [c for c in Xtr.columns if c.endswith("_cnt")]
Xte[cnt] = Xte[cnt].fillna(0).astype("uint8")

y = target.set_index("id").reindex(Xtr.index)["flag"].astype("int8")
print(f"Train: {Xtr.shape}  Test: {Xte.shape}")

In [ ]:
LGB_PARAMS = dict(
    objective="binary", metric="auc",
    learning_rate=0.02, num_leaves=96,
    feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1,
    min_child_samples=150, lambda_l1=0.5, lambda_l2=1.0,
    n_jobs=-1, verbose=-1,
)
N_FOLDS_GBM = 5
SEEDS_GBM   = [42, 137, 2025]

all_oofs_gbm, all_test_gbm = [], []

for seed in SEEDS_GBM:
    oof  = np.zeros(len(Xtr))
    tprd = np.zeros(len(Xte))
    skf  = StratifiedKFold(N_FOLDS_GBM, shuffle=True, random_state=seed)

    fold_bar = tqdm(skf.split(Xtr, y), total=N_FOLDS_GBM,
                    desc=f"LGB seed={seed}", leave=True)
    for fold, (tr, va) in enumerate(fold_bar, 1):
        m = lgb.train(
            {**LGB_PARAMS, "seed": seed},
            lgb.Dataset(Xtr.iloc[tr], y.iloc[tr]),
            num_boost_round=10000,
            valid_sets=[lgb.Dataset(Xtr.iloc[va], y.iloc[va])],
            callbacks=[lgb.early_stopping(400, verbose=False),
                       lgb.log_evaluation(99999)],
        )
        oof[va] = m.predict(Xtr.iloc[va], num_iteration=m.best_iteration)
        tprd   += m.predict(Xte,          num_iteration=m.best_iteration) / N_FOLDS_GBM
        auc     = roc_auc_score(y.iloc[va], oof[va])
        fold_bar.set_postfix(fold=fold, auc=f"{auc:.4f}", iter=m.best_iteration)

    oof_auc = roc_auc_score(y, oof)
    print(f"  seed={seed}  OOF AUC={oof_auc:.5f}")
    all_oofs_gbm.append(oof)
    all_test_gbm.append(tprd)

gbm_oof  = np.mean([rankdata(o)/len(o) for o in all_oofs_gbm], axis=0)
gbm_test = np.mean([rankdata(t)/len(t) for t in all_test_gbm], axis=0)
print(f"\nGBM ensemble OOF AUC = {roc_auc_score(y, gbm_oof):.5f}")

np.save(f"{WORK_DIR}/oof_gbm.npy",  np.vstack([Xtr.index.values.astype(float), gbm_oof]))
np.save(f"{WORK_DIR}/test_gbm.npy", np.vstack([Xte.index.values.astype(float), gbm_test]))

sub_gbm = sample[["id"]].copy()
sub_gbm["flag"] = sub_gbm["id"].map(
    pd.Series(gbm_test, index=Xte.index)).fillna(0.5)
sub_gbm.to_csv(f"{WORK_DIR}/submission_gbm.csv", index=False)
print("Saved: submission_gbm.csv  oof_gbm.npy  test_gbm.npy")

In [ ]:
MAX_LEN   = 20
VOCAB_CAP = 126

def build_sequences(df, feat_cols, vocab):
    df = df.sort_values(["id","rn"], kind="stable")
    ids = df["id"].values
    uniq_ids, first_pos, counts = np.unique(ids, return_index=True, return_counts=True)
    n_ids, n_feat = len(uniq_ids), len(feat_cols)

    pos  = (np.arange(len(df), dtype=np.int64) - np.repeat(first_pos, counts)).astype(np.int32)
    kfr  = np.repeat(np.maximum(counts - MAX_LEN, 0), counts).astype(np.int32)
    keep = pos >= kfr
    eff  = np.minimum(counts, MAX_LEN).astype(np.int16)
    tpos = (pos - kfr)

    gi = np.repeat(np.arange(n_ids, dtype=np.int32), counts)
    g  = gi[keep];  p = tpos[keep].astype(np.int64)
    del pos, kfr, tpos, gi, first_pos, ids; gc.collect()

    seq = np.zeros((n_ids, MAX_LEN, n_feat), dtype=np.int8)
    for k, c in enumerate(tqdm(feat_cols, desc="seq cols", leave=False)):
        v = np.clip(df[c].values.astype(np.int32), 0, vocab[c]-1) + 1
        seq[g, p, k] = v[keep].astype(np.int8)
        del v

    del df, keep, counts, g, p; gc.collect()
    return uniq_ids, seq, eff

# feat_cols берём из уже загруженного train (не перечитываем файл)
feat_cols = [c for c in train.columns if c != "id"]

print("Reloading raw data for sequences...")
train_raw = load_parts("train_data")
test_raw  = load_parts("test_data")

vocab = {c: min(int(max(train_raw[c].max(), test_raw[c].max())) + 1, VOCAB_CAP)
         for c in tqdm(feat_cols, desc="vocab", leave=False)}
vocab_sizes = [vocab[c]+1 for c in feat_cols]

print("Building train sequences...")
tr_ids, Xtr_seq, Ltr = build_sequences(train_raw, feat_cols, vocab)
del train_raw; gc.collect()
print("Building test sequences...")
te_ids, Xte_seq, Lte = build_sequences(test_raw, feat_cols, vocab)
del test_raw; gc.collect()

print(f"Train: {Xtr_seq.shape}  {Xtr_seq.nbytes/1e6:.0f} MB")
print(f"Test:  {Xte_seq.shape}  {Xte_seq.nbytes/1e6:.0f} MB")

y_nn = target.set_index("id")["flag"].reindex(tr_ids).values.astype(np.float32)

In [ ]:
class CreditGRU(nn.Module):
    def __init__(self, vocab_sizes, hidden=160, emb_max=16):
        super().__init__()
        self.embs = nn.ModuleList([
            nn.Embedding(vs, min(emb_max, (vs+1)//2+1), padding_idx=0)
            for vs in vocab_sizes
        ])
        emb_dim = sum(e.embedding_dim for e in self.embs)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.gru  = nn.GRU(hidden, hidden, num_layers=2, batch_first=True,
                           dropout=0.15, bidirectional=True)
        gout = hidden * 2
        self.attn = nn.Sequential(
            nn.Linear(gout, gout//2), nn.Tanh(), nn.Linear(gout//2, 1))
        self.head = nn.Sequential(
            nn.Linear(gout*3, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Dropout(0.25), nn.Linear(hidden, 1))

    def forward(self, x, lengths):
        h   = self.proj(torch.cat([e(x[:,:,k]) for k,e in enumerate(self.embs)], -1))
        out, _ = self.gru(h)
        L   = x.size(1)
        msk = torch.arange(L, device=x.device)[None,:] < lengths[:,None]
        sc  = self.attn(out).squeeze(-1).masked_fill(~msk, float("-inf"))
        ap  = (out * torch.softmax(sc,1).unsqueeze(-1)).sum(1)
        m   = msk.unsqueeze(-1).float()
        mp  = (out*m).sum(1) / m.sum(1).clamp(min=1)
        lp  = out[torch.arange(out.size(0), device=x.device),
                  (lengths-1).clamp(min=0)]
        return self.head(torch.cat([ap, mp, lp], -1)).squeeze(-1)

In [ ]:
N_FOLDS_NN = 5
EPOCHS     = 16
BATCH      = 2048
LR         = 2e-3
HIDDEN     = 160
EMB_MAX    = 16
PATIENCE   = 3
SEED_NN    = 42

torch.manual_seed(SEED_NN); np.random.seed(SEED_NN)

def predict_nn(model, X, L):
    model.eval()
    dl  = DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(L)),
                     batch_size=BATCH*2, shuffle=False)
    out = []
    with torch.no_grad():
        for xb, lb in dl:
            out.append(torch.sigmoid(
                model(xb.long().to(DEVICE), lb.long().to(DEVICE))).cpu().numpy())
    return np.concatenate(out)


def train_fold_nn(Xtr, Ltr, ytr, Xva, Lva, yva):
    model = CreditGRU(vocab_sizes, HIDDEN, EMB_MAX).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR, epochs=EPOCHS,
        steps_per_epoch=int(np.ceil(len(Xtr)/BATCH)), pct_start=0.15)
    pw    = torch.tensor([(len(ytr)-ytr.sum())/max(ytr.sum(),1)], device=DEVICE)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
    tr_dl = DataLoader(
        TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(Ltr),
                      torch.from_numpy(ytr)),
        batch_size=BATCH, shuffle=True,
        num_workers=2, pin_memory=(DEVICE=="cuda"), persistent_workers=True)
    va_dl = DataLoader(
        TensorDataset(torch.from_numpy(Xva), torch.from_numpy(Lva)),
        batch_size=BATCH*2, shuffle=False)

    best_auc, best_state, bad = 0.0, None, 0
    ep_bar = tqdm(range(EPOCHS), desc="epochs", leave=False)
    for ep in ep_bar:
        model.train()
        for xb, lb, yb in tr_dl:
            opt.zero_grad()
            loss = lossf(model(xb.long().to(DEVICE, non_blocking=True),
                               lb.long().to(DEVICE, non_blocking=True)),
                         yb.float().to(DEVICE, non_blocking=True))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step(); sched.step()
        model.eval()
        preds = []
        with torch.no_grad():
            for xb, lb in va_dl:
                preds.append(torch.sigmoid(
                    model(xb.long().to(DEVICE), lb.long().to(DEVICE))).cpu().numpy())
        auc = roc_auc_score(yva, np.concatenate(preds))
        ep_bar.set_postfix(auc=f"{auc:.4f}", best=f"{best_auc:.4f}")
        if auc > best_auc:
            best_auc  = auc
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return model, best_auc


oof_nn   = np.zeros(len(tr_ids))
te_preds = []
skf      = StratifiedKFold(N_FOLDS_NN, shuffle=True, random_state=SEED_NN)

fold_bar = tqdm(skf.split(Xtr_seq, y_nn), total=N_FOLDS_NN, desc="GRU folds")
for fold, (tr, va) in enumerate(fold_bar, 1):
    t0 = time.time()
    model, best = train_fold_nn(
        Xtr_seq[tr], Ltr[tr], y_nn[tr],
        Xtr_seq[va], Lva[va] if False else Ltr[va], y_nn[va])
    oof_nn[va] = predict_nn(model, Xtr_seq[va], Ltr[va])
    te_preds.append(predict_nn(model, Xte_seq, Lte))
    auc = roc_auc_score(y_nn[va], oof_nn[va])
    fold_bar.set_postfix(fold=fold, auc=f"{auc:.4f}",
                         best=f"{best:.4f}", sec=f"{time.time()-t0:.0f}")
    del model; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

nn_test = np.mean([rankdata(p)/len(p) for p in te_preds], axis=0)
print(f"GRU OOF AUC = {roc_auc_score(y_nn, oof_nn):.5f}")

np.save(f"{WORK_DIR}/oof_nn.npy",  np.vstack([tr_ids.astype(float), oof_nn]))
np.save(f"{WORK_DIR}/test_nn.npy", np.vstack([te_ids.astype(float), nn_test]))

sub_nn = sample[["id"]].copy()
sub_nn["flag"] = sub_nn["id"].map(
    pd.Series(nn_test, index=te_ids)).fillna(0.5)
sub_nn.to_csv(f"{WORK_DIR}/submission_nn.csv", index=False)
print("Saved: submission_nn.csv  oof_nn.npy  test_nn.npy")

In [ ]:
def load_pred(path):
    arr = np.load(path)
    return pd.Series(arr[1], index=arr[0].astype(np.int64))

oof_nn_s  = load_pred(f"{WORK_DIR}/oof_nn.npy")
oof_gbm_s = load_pred(f"{WORK_DIR}/oof_gbm.npy")
tgt       = target.set_index("id")["flag"]

ids  = oof_nn_s.index.intersection(oof_gbm_s.index).intersection(tgt.index)
y_e  = tgt.reindex(ids).values
rnn  = rankdata(oof_nn_s.reindex(ids).values)  / len(ids)
rgbm = rankdata(oof_gbm_s.reindex(ids).values) / len(ids)

auc_nn  = roc_auc_score(y_e, rnn)
auc_gbm = roc_auc_score(y_e, rgbm)
print(f"OOF AUC  NN={auc_nn:.5f}  GBM={auc_gbm:.5f}")

best_w, best_auc = 0.5, 0.0
for w in tqdm(np.linspace(0,1,101), desc="weight search", leave=False):
    auc = roc_auc_score(y_e, w*rnn + (1-w)*rgbm)
    if auc > best_auc:
        best_auc, best_w = auc, w

print(f"Best weight NN={best_w:.2f}  Ensemble OOF AUC={best_auc:.5f}  (+{best_auc-max(auc_nn,auc_gbm):.5f})")

te_nn_s  = load_pred(f"{WORK_DIR}/test_nn.npy")
te_gbm_s = load_pred(f"{WORK_DIR}/test_gbm.npy")
tids_e   = te_nn_s.index.intersection(te_gbm_s.index)
blend    = (best_w       * rankdata(te_nn_s.reindex(tids_e).values)  / len(tids_e)
          + (1-best_w)   * rankdata(te_gbm_s.reindex(tids_e).values) / len(tids_e))
pred_e   = pd.Series(blend, index=tids_e)

sub = sample[["id"]].copy()
sub["flag"] = sub["id"].map(pred_e).fillna(pred_e.mean())
sub.to_csv(f"{WORK_DIR}/submission_final.csv", index=False)
print(f"Saved: submission_final.csv  ({len(sub):,} rows)")
sub.head(10)